# Download the encrypted dataset

Let's download the dataset. We will download it from a public Azure storage blob.

The dataset is encrypted and also the connection with the storage is secured, so here we show **data in transit** security.

This means that if an attacker tries to sniff the network and captures this data while it is travelling, no data will be actually leaked since data is encrypted and only the sender and recipient can decrypt it.

Create dowload folder and install required packages

In [ ]:
! mkdir -p downloaded_datasets
! pip install azure.storage.blob > /dev/null

Fetch the Azure Shared Access Signature (SAS) string. The SAS per se is not really something private, in a real world scenario one would use a connection string, which cannot be shared publicly.

The SAS string is uploaded as sealed secret into this pod namespace, so if everything went well and attestation was successful, we should find the key under `/sealed/azure-value/azure-sas`:

In [ ]:
try:
    with open('/sealed/azure-value/azure-sas', 'r') as file:
        sas = file.read()
        print(sas)
except FileNotFoundError:
    print("File not found.")
except Exception as e:
    print(f"An error occurred: {e}")

Now that everything is ready, dowload the dataset

In [ ]:
import os
from azure.storage.blob import BlobClient

container_name = "data"
blob_name = "dataset1.csv.enc"
download_file_path = "downloaded_datasets/dataset1.csv.enc"
sas = 'NOT_FOUND'

try:
    with open('/sealed/azure-value/azure-sas', 'r') as file:
        sas = file.read()
except FileNotFoundError:
    print("File not found.")
except Exception as e:
    print(f"An error occurred: {e}")

if sas == 'NOT_FOUND':
    raise Exception("AZURE_ACCOUNT_SAS is not defined, cannot download the blob")

blob_client = BlobClient.from_blob_url("https://encrypteddatasets.blob.core.windows.net/" + container_name + "/" + blob_name + "?" + sas)

print(f"Downloading blob to: {download_file_path}")
with open(download_file_path, "wb") as download_file:
    blob_data = blob_client.download_blob()
    blob_data.readinto(download_file)
    print(f"Downloaded {download_file_path}")


print("Download complete!")

Let's check what's inside the dowloaded dataset:

In [ ]:
! head $download_file_path

As we can see, it's encrypted so we can't read any data!